In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql.window import Window
from delta.tables import DeltaTable

In [0]:
SILVER_PATH = "abfss://silver@pravdatalake.dfs.core.windows.net"
GOLD_PATH = "abfss://gold@pravdatalake.dfs.core.windows.net"
GOLD_TABLE_PATH = f"{GOLD_PATH}/dim_trim"
GOLD_TABLE_NAME = "vehicle_sales.gold.dim_trim"

In [0]:
silver_trim = spark.read.format("delta").load(f"{SILVER_PATH}/trim_table")

In [0]:
latest_window = Window.partitionBy("Genmodel_ID", "Trim").orderBy(col("Year").desc())

dim_trim_updates = (
    silver_trim
    .withColumn("rn", row_number().over(latest_window))
    .filter(col("rn") == 1)
    .drop("rn")
    .select("Genmodel_ID", "Trim", "Maker", "Genmodel", "Year",
            "Price", "Gas_emission", "Fuel_type", "Engine_size")
)

In [0]:
dim_trim_updates.display()

####Data Quality checks

In [0]:
row_count = dim_trim_updates.count()

In [0]:
null_key_count = dim_trim_updates.filter(col("Genmodel_ID").isNull() | col("Trim").isNull()).count()

In [0]:
duplicate_key_count = dim_trim_updates.groupBy("Genmodel_ID", "Trim").count().filter("count > 1").count()

In [0]:
print(f"row count: {row_count}")
print(f"null key count: {null_key_count}")
print(f"duplicate key count: {duplicate_key_count}")

In [0]:
assert null_key_count == 0, "Genmodel_ID/Trim should never be null in dim_trim"
assert duplicate_key_count == 0, "(Genmodel_ID, Trim) should be unique after collapsing to latest year"

In [0]:
# --- SCD Type 2: keep full version history ---
# Step 1: close out any current row whose tracked attributes changed.
# Step 2: insert a new current row for anything new or changed.
# Attributes we track history on:
#   Price, Gas_emission, Fuel_type, Engine_size
if DeltaTable.isDeltaTable(spark, GOLD_TABLE_PATH):
 
    dim_trim_table = DeltaTable.forPath(spark, GOLD_TABLE_PATH)
 
    change_condition = """
        t.Price <> s.Price
        OR t.Gas_emission <> s.Gas_emission
        OR t.Fuel_type <> s.Fuel_type
        OR t.Engine_size <> s.Engine_size
    """
 
    (dim_trim_table.alias("t")
        .merge(
            dim_trim_updates.alias("s"),
            "t.Genmodel_ID = s.Genmodel_ID AND t.Trim = s.Trim AND t.is_current = true"
        )
        .whenMatchedUpdate(
            condition=change_condition,
            set={"is_current": "false", "effective_end_date": "current_date()"}
        )
        .execute())
 
    current_rows = (
        spark.read.format("delta").load(GOLD_TABLE_PATH)
        .filter("is_current = true")
        .select("Genmodel_ID", "Trim", "Price", "Gas_emission", "Fuel_type", "Engine_size")
    )
 
    to_insert = (
        dim_trim_updates.alias("s")
        .join(current_rows.alias("c"), ["Genmodel_ID", "Trim"], "left")
        .where(
            col("c.Genmodel_ID").isNull()
            | (col("s.Price") != col("c.Price"))
            | (col("s.Gas_emission") != col("c.Gas_emission"))
            | (col("s.Fuel_type") != col("c.Fuel_type"))
            | (col("s.Engine_size") != col("c.Engine_size"))
        )
        .select("s.*")
        .withColumn("effective_start_date", current_date())
        .withColumn("effective_end_date", lit(None).cast(DateType()))
        .withColumn("is_current", lit(True))
    )
 
    insert_count = to_insert.count()
    print(f"new/changed trim versions to insert: {insert_count}")
 
    if insert_count > 0:
        to_insert.write \
            .format("delta") \
            .mode("append") \
            .option("mergeSchema", "true") \
            .save(GOLD_TABLE_PATH)
 
else:
 
    initial_load = (
        dim_trim_updates
        .withColumn("effective_start_date", current_date())
        .withColumn("effective_end_date", lit(None).cast(DateType()))
        .withColumn("is_current", lit(True))
    )
 
    initial_load.write \
        .format("delta") \
        .mode("overwrite") \
        .save(GOLD_TABLE_PATH)
 

In [0]:
spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {GOLD_TABLE_NAME}
    USING DELTA
    LOCATION '{GOLD_TABLE_PATH}'
""")

In [0]:
spark.sql(f"OPTIMIZE {GOLD_TABLE_NAME} ZORDER BY (Genmodel_ID)")

In [0]:
# --- post-write validation: exactly one is_current row per business key ---
current_duplicate_count = (
    spark.read.format("delta").load(GOLD_TABLE_PATH)
    .filter("is_current = true")
    .groupBy("Genmodel_ID", "Trim")
    .count()
    .filter("count > 1")
    .count()
)

In [0]:
print(f"business keys with more than one current row: {current_duplicate_count}")

In [0]:
assert current_duplicate_count == 0, "dim_trim should have exactly one is_current row per (Genmodel_ID, Trim)"